
# Predictive Customer Value

## Business Problem

Our Customer Intelligence Platform currently estimates whether customers will make another purchase within 60 days. However, repeat-purchase likelihood does not tell us how much additional merchandise value those customers might generate.

The business needs to understand whether predicted customer monetary value can support better customer prioritisation beyond repeat-purchase propensity alone.

## Business Question

At the time of a customer's first recorded purchase, how much additional delivered merchandise value can we expect that customer to generate within the following 60 days?

## Modelling Objective

Develop and evaluate a predictive model for additional 60-day merchandise value, excluding the customer's first recorded order.

The initial modelling approach will investigate:

- The probability of generating subsequent delivered merchandise value.
- The amount of additional merchandise value conditional on a qualifying repeat purchase.
- The combined expected additional merchandise value per customer.

## Success Criteria

The model must:

- Use only features available at the prediction point.
- Outperform simple baseline approaches on relevant monetary-value metrics.
- Demonstrate useful customer-value ranking, not just repeat-purchase classification.
- Be evaluated chronologically to reflect future deployment conditions.
- Clearly communicate uncertainty and commercial limitations.

## Validation Requirements

Previous modelling work used chronological training and validation periods, and the earlier test cohort has already been examined.

Therefore, we must not describe that cohort as an untouched independent holdout.

Before model development, we will audit the available data and determine whether a genuinely independent future evaluation population exists.

## Scope and Limitations

The prediction target measures future delivered merchandise value within a fixed 60-day window.

It does not represent customer profitability, net revenue, or complete Customer Lifetime Value.

No production deployment or commercial intervention will be recommended without suitable validation.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# Locate the project directory and processed datasets
project_root = Path.cwd().parent
processed_dir = project_root / "data" / "processed"

# Load the existing feature engineering and customer intelligence datasets
customer_features = pd.read_pickle(
    processed_dir / "customer_features.pkl"
)

historical_intelligence = pd.read_parquet(
    processed_dir / "customer_intelligence_historical.parquet"
)

# Review dataset dimensions, columns and missing values
for dataset_name, df in {
    "Customer Features": customer_features,
    "Historical Customer Intelligence": historical_intelligence
}.items():
    print(f"\n{'=' * 60}")
    print(f"{dataset_name}: {df.shape[0]:,} rows × {df.shape[1]} columns")
    print(f"{'=' * 60}")

    display(pd.DataFrame({
        "Data Type": df.dtypes.astype(str),
        "Missing Values": df.isna().sum(),
        "Unique Values": df.nunique(dropna=True)
    }))


Customer Features: 78,100 rows × 29 columns


,Data Type,Missing Values,Unique Values
customer_unique_id,str,0,78100
order_id,str,0,78100
order_purchase_timestamp,datetime64[us],0,77894
customer_city,str,0,3923
customer_state,str,0,27
customer_zip_code_prefix,int64,0,14328
repeat_purchase_60d,int64,0,2
purchase_month,int32,0,12
purchase_day_of_week,int32,0,7
purchase_hour,int32,0,24



Historical Customer Intelligence: 78,100 rows × 9 columns


,Data Type,Missing Values,Unique Values
customer_unique_id,str,0,78100
order_id,str,0,78100
order_purchase_timestamp,datetime64[us],0,77894
first_order_item_value,float64,642,6514
repeat_purchase_propensity,float64,0,77785
propensity_segment,string,0,3
initial_value_segment,string,0,3
customer_profile,string,0,7
repeat_purchase_60d,int64,0,2


In [2]:
# Inspect the historical intelligence dataset and monetary target availability
# Do not use existing model scores as predictive features at this stage

print("HISTORICAL CUSTOMER INTELLIGENCE")
print(f"Rows: {len(historical_intelligence):,}")
print(f"Unique customers: {historical_intelligence['customer_unique_id'].nunique():,}")

display(pd.DataFrame({
    "Column": historical_intelligence.columns,
    "Data Type": historical_intelligence.dtypes.astype(str).values,
    "Missing Values": historical_intelligence.isna().sum().values
}))

# Identify available customer-value fields without assuming they exist
value_fields = [
    col for col in historical_intelligence.columns
    if any(term in col.lower() for term in [
        "revenue", "merchandise", "subsequent", "value_60d"
    ])
]

print("\nMONETARY TARGET COLUMNS FOUND:")
print(value_fields if value_fields else "None")

print("\nCUSTOMER IDENTIFIER RECONCILIATION")

feature_ids = set(customer_features["customer_unique_id"])
intelligence_ids = set(historical_intelligence["customer_unique_id"])

print("Customers in both datasets:", len(feature_ids & intelligence_ids))
print("Features only:", len(feature_ids - intelligence_ids))
print("Intelligence only:", len(intelligence_ids - feature_ids))

HISTORICAL CUSTOMER INTELLIGENCE
Rows: 78,100
Unique customers: 78,100


,Column,Data Type,Missing Values
0,customer_unique_id,str,0
1,order_id,str,0
2,order_purchase_timestamp,datetime64[us],0
3,first_order_item_value,float64,642
4,repeat_purchase_propensity,float64,0
5,propensity_segment,string,0
6,initial_value_segment,string,0
7,customer_profile,string,0
8,repeat_purchase_60d,int64,0



MONETARY TARGET COLUMNS FOUND:
None

CUSTOMER IDENTIFIER RECONCILIATION
Customers in both datasets: 78100
Features only: 0
Intelligence only: 0


In [3]:
# Load validated customer-value targets from Notebook 09
value_targets = pd.read_parquet(
    processed_dir / "customer_value_60d_targets.parquet"
)

# Retain only the future monetary outcome as our prediction target
modelling_data = customer_features.merge(
    value_targets[
        ["customer_unique_id", "subsequent_60d_value"]
    ],
    on="customer_unique_id",
    how="left",
    validate="one_to_one"
)

# Validate customer coverage and target consistency
assert len(modelling_data) == 78_100
assert modelling_data["customer_unique_id"].is_unique
assert modelling_data["subsequent_60d_value"].notna().all()

assert (
    modelling_data["subsequent_60d_value"] >= 0
).all()

# Verify consistency with our existing repeat-purchase outcome
target_alignment = pd.crosstab(
    modelling_data["repeat_purchase_60d"],
    modelling_data["subsequent_60d_value"] > 0,
    rownames=["Repeat-purchase label"],
    colnames=["Generated subsequent value"],
    margins=True
)

print("MODELLING DATASET")
print(f"Customers: {len(modelling_data):,}")
print(f"Columns: {modelling_data.shape[1]}")

print("\nTARGET ALIGNMENT")
display(target_alignment)

print("\nMONETARY TARGET DISTRIBUTION")
display(
    modelling_data["subsequent_60d_value"]
    .describe(percentiles=[0.50, 0.90, 0.95, 0.99])
    .round(2)
)

MODELLING DATASET
Customers: 78,100
Columns: 30

TARGET ALIGNMENT


Generated subsequent value,False,True,All
Repeat-purchase label,,,
0,76511,0,76511
1,0,1589,1589
All,76511,1589,78100



MONETARY TARGET DISTRIBUTION


count    78100.00
mean         2.71
std         31.53
min          0.00
50%          0.00
90%          0.00
95%          0.00
99%         85.00
max       2589.00
Name: subsequent_60d_value, dtype: float64

In [4]:
# Audit chronological populations and monetary target availability
# Preserve the date boundaries from the existing repeat-purchase analysis

modelling_data["order_purchase_timestamp"] = pd.to_datetime(
    modelling_data["order_purchase_timestamp"]
)

purchase_date = modelling_data["order_purchase_timestamp"]

conditions = [
    purchase_date <= pd.Timestamp("2017-06-30 23:59:59"),

    purchase_date.between(
        "2017-10-01",
        "2017-12-31 23:59:59"
    ),

    purchase_date >= pd.Timestamp("2018-04-01")
]

labels = [
    "Original training period",
    "Original validation period",
    "Previously examined test period"
]

modelling_data["chronological_population"] = np.select(
    conditions,
    labels,
    default="Outside original splits"
)

chronological_audit = (
    modelling_data
    .groupby("chronological_population", as_index=False)
    .agg(
        customers=("customer_unique_id", "count"),
        positive_value_customers=(
            "subsequent_60d_value",
            lambda s: (s > 0).sum()
        ),
        total_subsequent_value=("subsequent_60d_value", "sum"),
        average_subsequent_value=("subsequent_60d_value", "mean")
    )
)

chronological_audit["positive_rate_pct"] = (
    100
    * chronological_audit["positive_value_customers"]
    / chronological_audit["customers"]
)

display(chronological_audit.round(2))

assert chronological_audit["customers"].sum() == 78_100

,chronological_population,customers,positive_value_customers,total_subsequent_value,average_subsequent_value,positive_rate_pct
0,Original training period,14554,321,39478.82,2.71,2.21
1,Original validation period,17257,331,46370.94,2.69,1.92
2,Outside original splits,32640,738,93744.01,2.87,2.26
3,Previously examined test period,13649,199,31701.42,2.32,1.46


In [5]:
# Examine monetary-target availability across first-purchase months
# Identify temporal changes before designing training and validation periods

modelling_data["first_purchase_month"] = (
    modelling_data["order_purchase_timestamp"]
    .dt.to_period("M")
)

monthly_target_audit = (
    modelling_data
    .groupby("first_purchase_month", as_index=False)
    .agg(
        customers=("customer_unique_id", "count"),
        positive_value_customers=(
            "subsequent_60d_value",
            lambda s: (s > 0).sum()
        ),
        total_subsequent_value=("subsequent_60d_value", "sum"),
        average_subsequent_value=("subsequent_60d_value", "mean")
    )
)

monthly_target_audit["positive_rate_pct"] = (
    100
    * monthly_target_audit["positive_value_customers"]
    / monthly_target_audit["customers"]
)

monthly_target_audit["average_value_per_repeater"] = (
    monthly_target_audit["total_subsequent_value"]
    / monthly_target_audit["positive_value_customers"].replace(0, np.nan)
)

display(monthly_target_audit.round(2))

print("\nMONTHLY TARGET AUDIT")

print(
    "Total customers:",
    f"{monthly_target_audit['customers'].sum():,}"
)

print(
    "Customers with positive value:",
    f"{monthly_target_audit['positive_value_customers'].sum():,}"
)

print(
    "Earliest cohort:",
    monthly_target_audit["first_purchase_month"].min()
)

print(
    "Latest cohort:",
    monthly_target_audit["first_purchase_month"].max()
)

assert monthly_target_audit["customers"].sum() == 78_100
assert monthly_target_audit["positive_value_customers"].sum() == 1_589

C:\Users\oades\AppData\Local\Temp\ipykernel_32460\893574085.py:34: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  display(monthly_target_audit.round(2))


,first_purchase_month,customers,positive_value_customers,total_subsequent_value,average_subsequent_value,positive_rate_pct,average_value_per_repeater
0,2016-09,4,0,0.00,0.00,0.00,NaN
1,2016-10,321,3,470.95,1.47,0.93,156.98
2,2016-12,1,1,10.90,10.90,100.00,10.90
3,2017-01,762,27,2517.00,3.30,3.54,93.22
4,2017-02,1751,27,3281.49,1.87,1.54,121.54
5,2017-03,2635,53,5245.09,1.99,2.01,98.96
6,2017-04,2350,42,7581.59,3.23,1.79,180.51
7,2017-05,3594,93,11158.60,3.10,2.59,119.98
8,2017-06,3136,75,9213.20,2.94,2.39,122.84
9,2017-07,3893,95,11717.57,3.01,2.44,123.34



MONTHLY TARGET AUDIT
Total customers: 78,100
Customers with positive value: 1,589
Earliest cohort: 2016-09
Latest cohort: 2018-06


In [6]:
# Audit candidate predictive features and identify potential leakage risks
# This is a structural review, not yet an approved feature selection

excluded_columns = {
    "customer_unique_id",
    "order_id",
    "repeat_purchase_60d",
    "subsequent_60d_value"
}

feature_audit = pd.DataFrame({
    "feature": [
        col for col in modelling_data.columns
        if col not in excluded_columns
    ]
})

feature_audit["data_type"] = feature_audit["feature"].map(
    lambda col: str(modelling_data[col].dtype)
)

feature_audit["missing_values"] = feature_audit["feature"].map(
    lambda col: int(modelling_data[col].isna().sum())
)

feature_audit["potential_timing_risk"] = (
    feature_audit["feature"]
    .str.contains(
        "payment|freight|basket|item|seller|product|category",
        case=False,
        regex=True
    )
)

feature_audit["review_status"] = np.where(
    feature_audit["potential_timing_risk"],
    "Verify availability at prediction time",
    "Review definition and source"
)

display(feature_audit)

print("\nFEATURE AUDIT SUMMARY")
print(f"Candidate fields: {len(feature_audit)}")
print(
    "Fields requiring explicit timing verification:",
    feature_audit["potential_timing_risk"].sum()
)

,feature,data_type,missing_values,potential_timing_risk,review_status
0,order_purchase_timestamp,datetime64[us],0,False,Review definition and source
1,customer_city,str,0,False,Review definition and source
2,customer_state,str,0,False,Review definition and source
3,customer_zip_code_prefix,int64,0,False,Review definition and source
4,purchase_month,int32,0,False,Review definition and source
5,purchase_day_of_week,int32,0,False,Review definition and source
6,purchase_hour,int32,0,False,Review definition and source
7,first_order_item_count,float64,642,True,Verify availability at prediction time
8,first_order_product_count,float64,642,True,Verify availability at prediction time
9,first_order_seller_count,float64,642,True,Verify availability at prediction time



FEATURE AUDIT SUMMARY
Candidate fields: 28
Fields requiring explicit timing verification: 13


In [7]:
# List payment-related features in the saved modelling dataset
# Confirm their exact names before completing the provenance audit

payment_related_columns = [
    column
    for column in modelling_data.columns
    if (
        "payment" in column.lower()
        or "installment" in column.lower()
        or column.startswith("uses_")
    )
]

print("PAYMENT-RELATED FEATURES")

for column in payment_related_columns:
    print(column)

print(
    "\nTotal payment-related fields:",
    len(payment_related_columns)
)

PAYMENT-RELATED FEATURES
uses_boleto
uses_credit_card
uses_debit_card
uses_voucher
first_order_max_installments
first_order_payment_entries
payment_method_count

Total payment-related fields: 7


In [8]:
# Identify remaining features requiring provenance review
# Exclude identifiers, targets, payment features and verified basket features

basket_features = {
    "first_order_item_count",
    "first_order_product_count",
    "first_order_seller_count",
    "first_order_item_value"
}

payment_features = set(payment_related_columns)

non_predictive_fields = {
    "customer_unique_id",
    "order_id",
    "repeat_purchase_60d",
    "subsequent_60d_value",
    "chronological_population",
    "first_purchase_month"
}

remaining_features = [
    column
    for column in modelling_data.columns
    if column not in (
        basket_features
        | payment_features
        | non_predictive_fields
    )
]

print("REMAINING FEATURES TO REVIEW")

for feature in remaining_features:
    print(feature)

print(f"\nTotal remaining fields: {len(remaining_features)}")

REMAINING FEATURES TO REVIEW
order_purchase_timestamp
customer_city
customer_state
customer_zip_code_prefix
purchase_month
purchase_day_of_week
purchase_hour
first_order_freight_value
first_order_total_basket_value
first_order_total_basket_value_log
first_order_category_count
first_order_primary_category
primary_category_value
first_order_primary_category_share
customer_city_state

Total remaining fields: 15


In [9]:
# Define candidate features for predictive customer-value modelling
# Exclude identifiers, historical outcomes, and evaluation labels

numeric_features = [
    "first_order_item_count",
    "first_order_product_count",
    "first_order_seller_count",
    "first_order_item_value",
    "first_order_freight_value",
    "first_order_total_basket_value_log",
    "purchase_month",
    "purchase_day_of_week",
    "purchase_hour",
    "first_order_max_installments",
    "payment_method_count",
    "first_order_category_count",
    "primary_category_value",
    "first_order_primary_category_share",
]

categorical_features = [
    "customer_state",
    "first_order_primary_category",
    "uses_credit_card",
    "uses_boleto",
    "uses_debit_card",
    "uses_voucher",
]

feature_columns = numeric_features + categorical_features

X = modelling_data[feature_columns].copy()
y = modelling_data["subsequent_60d_value"].copy()

# Verify feature availability and target integrity
assert X.columns.is_unique
assert len(X) == len(y) == 78_100
assert y.notna().all()
assert (y >= 0).all()

print("INITIAL MODELLING FEATURE SET")
print(f"Customers: {len(X):,}")
print(f"Numeric features: {len(numeric_features)}")
print(f"Categorical features: {len(categorical_features)}")
print(f"Total features: {len(feature_columns)}")

print("\nMISSING FEATURE VALUES")
display(
    X.isna()
    .sum()
    .loc[lambda s: s > 0]
    .sort_values(ascending=False)
)

print("\nTARGET SUMMARY")
print(f"Customers with positive value: {(y > 0).sum():,}")
print(f"Customers with zero value: {(y == 0).sum():,}")
print(f"Average subsequent value: R${y.mean():.2f}")

INITIAL MODELLING FEATURE SET
Customers: 78,100
Numeric features: 14
Categorical features: 6
Total features: 20

MISSING FEATURE VALUES


first_order_primary_category_share    1903
first_order_primary_category          1903
primary_category_value                1903
first_order_item_count                 642
first_order_freight_value              642
first_order_product_count              642
first_order_seller_count               642
first_order_item_value                 642
first_order_total_basket_value_log     642
first_order_category_count             642
payment_method_count                     1
first_order_max_installments             1
uses_credit_card                         1
uses_boleto                              1
uses_debit_card                          1
uses_voucher                             1
dtype: int64


TARGET SUMMARY
Customers with positive value: 1,589
Customers with zero value: 76,511
Average subsequent value: R$2.71


In [16]:
# Define chronological modelling populations
# Keep the original temporal boundaries for comparability

purchase_dates = pd.to_datetime(
    modelling_data["order_purchase_timestamp"]
)

train_mask = (
    purchase_dates <= pd.Timestamp("2017-06-30 23:59:59")
)

validation_mask = purchase_dates.between(
    "2017-10-01",
    "2017-12-31 23:59:59"
)

evaluation_mask = (
    purchase_dates >= pd.Timestamp("2018-04-01")
)

# Separate features and monetary targets
X_train = X.loc[train_mask].copy()
X_val = X.loc[validation_mask].copy()
X_eval = X.loc[evaluation_mask].copy()

y_train = y.loc[train_mask].copy()
y_val = y.loc[validation_mask].copy()
y_eval = y.loc[evaluation_mask].copy()

# Check chronology and population separation
assert (
    purchase_dates.loc[train_mask].max()
    < purchase_dates.loc[validation_mask].min()
)

assert (
    purchase_dates.loc[validation_mask].max()
    < purchase_dates.loc[evaluation_mask].min()
)

assert not (
    train_mask & validation_mask
).any()

assert not (
    train_mask & evaluation_mask
).any()

assert not (
    validation_mask & evaluation_mask
).any()

# Summarise monetary outcomes in each population
split_summary = pd.DataFrame({
    "Population": [
        "Training",
        "Development validation",
        "Retrospective evaluation"
    ],
    "Customers": [
        len(y_train),
        len(y_val),
        len(y_eval)
    ],
    "Positive Value Customers": [
        (y_train > 0).sum(),
        (y_val > 0).sum(),
        (y_eval > 0).sum()
    ],
    "Average Subsequent Value (R$)": [
        y_train.mean(),
        y_val.mean(),
        y_eval.mean()
    ]
})

split_summary["Positive Rate (%)"] = (
    100
    * split_summary["Positive Value Customers"]
    / split_summary["Customers"]
)

display(split_summary.round(3))

print(
    "\nCustomers outside these splits:",
    len(modelling_data)
    - len(y_train)
    - len(y_val)
    - len(y_eval)
)

,Population,Customers,Positive Value Customers,Average Subsequent Value (R$),Positive Rate (%)
0,Training,14554,321,2.713,2.206
1,Development validation,17257,331,2.687,1.918
2,Retrospective evaluation,13649,199,2.323,1.458



Customers outside these splits: 32640


In [11]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Establish simple monetary-value prediction baselines
# All baseline parameters must come from the training population

training_mean_value = y_train.mean()

baseline_results = []

for population_name, actual_values in [
    ("Training", y_train),
    ("Development validation", y_val)
]:
    actual_values = actual_values.to_numpy()

    baseline_predictions = {
        "Predict zero": np.zeros(len(actual_values)),
        "Predict training mean": np.full(
            len(actual_values),
            training_mean_value
        )
    }

    for baseline_name, predictions in baseline_predictions.items():
        baseline_results.append({
            "Population": population_name,
            "Baseline": baseline_name,
            "Mean Absolute Error (R$)": mean_absolute_error(
                actual_values, predictions
            ),
            "Root Mean Squared Error (R$)": np.sqrt(
                mean_squared_error(actual_values, predictions)
            ),
            "Predicted Average Value (R$)": predictions.mean(),
            "Observed Average Value (R$)": actual_values.mean()
        })

baseline_summary = pd.DataFrame(baseline_results)

display(baseline_summary.round(3))

print(
    "\nTraining mean used for predictions:",
    f"R${training_mean_value:.4f}"
)

,Population,Baseline,Mean Absolute Error (R$),Root Mean Squared Error (R$),Predicted Average Value (R$),Observed Average Value (R$)
0,Training,Predict zero,2.713,36.707,0.000,2.713
1,Training,Predict training mean,5.305,36.607,2.713,2.713
2,Development validation,Predict zero,2.687,33.240,0.000,2.687
3,Development validation,Predict training mean,5.296,33.131,2.713,2.687



Training mean used for predictions: R$2.7126


In [12]:
# Establish a simple two-stage customer-value baseline
# Estimate both components using training data only

training_repeat_probability = (y_train > 0).mean()

training_positive_mean = y_train.loc[
    y_train > 0
].mean()

expected_value_baseline = (
    training_repeat_probability
    * training_positive_mean
)

print("TWO-STAGE BASELINE COMPONENTS")

print(
    "Training repeat probability:",
    f"{training_repeat_probability:.4%}"
)

print(
    "Average value among training repeat purchasers:",
    f"R${training_positive_mean:.2f}"
)

print(
    "Expected subsequent value per customer:",
    f"R${expected_value_baseline:.4f}"
)

print(
    "\nTraining population mean:",
    f"R${y_train.mean():.4f}"
)

assert np.isclose(
    expected_value_baseline,
    y_train.mean()
)

TWO-STAGE BASELINE COMPONENTS
Training repeat probability: 2.2056%
Average value among training repeat purchasers: R$122.99
Expected subsequent value per customer: R$2.7126

Training population mean: R$2.7126


In [14]:

# Define candidate features for predictive customer-value modelling
# Exclude identifiers, historical outcomes, and evaluation labels
# Treat binary payment-method indicators as numeric features

numeric_features = [
    "first_order_item_count",
    "first_order_product_count",
    "first_order_seller_count",
    "first_order_item_value",
    "first_order_freight_value",
    "first_order_total_basket_value_log",
    "purchase_month",
    "purchase_day_of_week",
    "purchase_hour",
    "first_order_max_installments",
    "payment_method_count",
    "first_order_category_count",
    "primary_category_value",
    "first_order_primary_category_share",
    "uses_credit_card",
    "uses_boleto",
    "uses_debit_card",
    "uses_voucher",
]

categorical_features = [
    "customer_state",
    "first_order_primary_category",
]

# Combine the selected candidate features
feature_columns = numeric_features + categorical_features

# Construct the predictive feature matrix and monetary target
X = modelling_data[feature_columns].copy()
y = modelling_data["subsequent_60d_value"].copy()

# Validate feature availability and target integrity
assert X.columns.is_unique
assert len(X) == len(y) == 78_100
assert y.notna().all()
assert (y >= 0).all()

# Ensure the numeric features contain valid numeric values
for column in numeric_features:
    assert pd.api.types.is_numeric_dtype(X[column]), (
        f"Non-numeric values detected in {column}"
    )

# Ensure categorical features contain only strings or missing values
for column in categorical_features:
    assert X[column].dropna().map(
        lambda value: isinstance(value, str)
    ).all(), f"Unexpected category type in {column}"

print("INITIAL MODELLING FEATURE SET")
print(f"Customers: {len(X):,}")
print(f"Numeric features: {len(numeric_features)}")
print(f"Categorical features: {len(categorical_features)}")
print(f"Total features: {len(feature_columns)}")

print("\nMISSING FEATURE VALUES")

display(
    X.isna()
    .sum()
    .loc[lambda s: s > 0]
    .sort_values(ascending=False)
)

print("\nTARGET SUMMARY")

print(
    "Customers with positive value:",
    f"{(y > 0).sum():,}"
)

print(
    "Customers with zero value:",
    f"{(y == 0).sum():,}"
)

print(
    "Average subsequent value:",
    f"R${y.mean():.2f}"
)


INITIAL MODELLING FEATURE SET
Customers: 78,100
Numeric features: 18
Categorical features: 2
Total features: 20

MISSING FEATURE VALUES


first_order_primary_category_share    1903
first_order_primary_category          1903
primary_category_value                1903
first_order_item_count                 642
first_order_freight_value              642
first_order_product_count              642
first_order_seller_count               642
first_order_item_value                 642
first_order_total_basket_value_log     642
first_order_category_count             642
payment_method_count                     1
first_order_max_installments             1
uses_credit_card                         1
uses_boleto                              1
uses_debit_card                          1
uses_voucher                             1
dtype: int64


TARGET SUMMARY
Customers with positive value: 1,589
Customers with zero value: 76,511
Average subsequent value: R$2.71


In [17]:

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Prepare numeric features using training-data statistics
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Prepare categorical features
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(
        strategy="constant",
        fill_value="Missing"
    )),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Apply preprocessing to the appropriate feature groups
preprocessor = ColumnTransformer([
    ("numeric", numeric_transformer, numeric_features),
    ("categorical", categorical_transformer, categorical_features)
])

# Train the repeat-purchase probability component
repeat_probability_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        C=0.1,
        class_weight=None,
        max_iter=2000,
        random_state=42
    ))
])

repeat_probability_model.fit(
    X_train,
    (y_train > 0).astype(int)
)

# Predict repeat-purchase probability for validation customers
validation_repeat_probability = (
    repeat_probability_model.predict_proba(X_val)[:, 1]
)

# Combine predicted probability with average spending among
# repeat purchasers, estimated from training customers only
validation_expected_value = (
    validation_repeat_probability * training_positive_mean
)

# Validate predictions
assert len(validation_expected_value) == len(y_val)
assert np.isfinite(validation_expected_value).all()
assert (validation_expected_value >= 0).all()

print("FIRST CUSTOMER-SPECIFIC VALUE MODEL")

print(
    "Average predicted repeat probability:",
    f"{validation_repeat_probability.mean():.4%}"
)

print(
    "Average predicted subsequent value:",
    f"R${validation_expected_value.mean():.4f}"
)

print(
    "Observed average subsequent value:",
    f"R${y_val.mean():.4f}"
)

print("\nPREDICTED VALUE DISTRIBUTION")

display(
    pd.Series(
        validation_expected_value,
        name="predicted_subsequent_value"
    ).describe(
        percentiles=[0.10, 0.25, 0.50, 0.75, 0.90, 0.99]
    ).round(3)
)


FIRST CUSTOMER-SPECIFIC VALUE MODEL
Average predicted repeat probability: 2.2930%
Average predicted subsequent value: R$2.8201
Observed average subsequent value: R$2.6871

PREDICTED VALUE DISTRIBUTION


count    17257.000
mean         2.820
std          2.198
min          0.295
10%          1.149
25%          1.568
50%          2.294
75%          3.395
90%          5.014
99%          9.983
max         70.926
Name: predicted_subsequent_value, dtype: float64

In [18]:

# Evaluate customer-specific monetary predictions against simple baselines
# Use development validation only; do not use the retrospective evaluation cohort

actual_values = y_val.to_numpy()

# Baselines use training-data statistics only
model_comparison_predictions = {
    "Predict zero": np.zeros(len(actual_values)),
    "Predict training mean": np.full(
        len(actual_values),
        training_mean_value
    ),
    "Customer-specific two-stage model": validation_expected_value
}

comparison_results = []

for model_name, predictions in model_comparison_predictions.items():
    comparison_results.append({
        "Model": model_name,
        "Mean Absolute Error (R$)": mean_absolute_error(
            actual_values, predictions
        ),
        "Root Mean Squared Error (R$)": np.sqrt(
            mean_squared_error(actual_values, predictions)
        ),
        "Predicted Average Value (R$)": predictions.mean(),
        "Observed Average Value (R$)": actual_values.mean(),
        "Average Prediction Bias (R$)": (
            predictions.mean() - actual_values.mean()
        )
    })

model_comparison_summary = pd.DataFrame(comparison_results)

display(
    model_comparison_summary.round(3)
)


,Model,Mean Absolute Error (R$),Root Mean Squared Error (R$),Predicted Average Value (R$),Observed Average Value (R$),Average Prediction Bias (R$)
0,Predict zero,2.687,33.240,0.000,2.687,-2.687
1,Predict training mean,5.296,33.131,2.713,2.687,0.025
2,Customer-specific two-stage model,5.352,33.134,2.820,2.687,0.133


In [19]:

# Evaluate customer-value ranking on development validation
# Assess whether higher predictions correspond to greater observed value

ranking_evaluation = pd.DataFrame({
    "predicted_value": validation_expected_value,
    "actual_value": y_val.to_numpy()
})

# Assign customers to ten groups ranked by predicted value
# Rank first to avoid duplicate quantile boundaries
ranking_evaluation["value_decile"] = pd.qcut(
    ranking_evaluation["predicted_value"].rank(method="first"),
    q=10,
    labels=range(1, 11)
).astype(int)

ranking_summary = (
    ranking_evaluation
    .groupby("value_decile", as_index=False)
    .agg(
        customers=("actual_value", "count"),
        average_predicted_value=("predicted_value", "mean"),
        average_actual_value=("actual_value", "mean"),
        total_actual_value=("actual_value", "sum"),
        customers_with_positive_value=(
            "actual_value",
            lambda s: (s > 0).sum()
        )
    )
)

ranking_summary["positive_value_rate_pct"] = (
    100
    * ranking_summary["customers_with_positive_value"]
    / ranking_summary["customers"]
)

# Decile 10 represents the highest predicted customer value
ranking_summary["value_decile_label"] = (
    "Decile " + ranking_summary["value_decile"].astype(str)
)

assert ranking_summary["customers"].sum() == len(y_val)

assert np.isclose(
    ranking_summary["total_actual_value"].sum(),
    y_val.sum()
)

display(ranking_summary.round(3))

print("\nTOP VERSUS BOTTOM DECILE")

top_decile = ranking_summary.loc[
    ranking_summary["value_decile"] == 10
].iloc[0]

bottom_decile = ranking_summary.loc[
    ranking_summary["value_decile"] == 1
].iloc[0]

print(
    "Top-decile average observed value:",
    f"R${top_decile['average_actual_value']:.2f}"
)

print(
    "Bottom-decile average observed value:",
    f"R${bottom_decile['average_actual_value']:.2f}"
)

print(
    "Top-decile share of total subsequent value:",
    f"{100 * top_decile['total_actual_value'] / y_val.sum():.2f}%"
)


,value_decile,customers,average_predicted_value,average_actual_value,total_actual_value,customers_with_positive_value,positive_value_rate_pct,value_decile_label
0,1,1726,0.933,3.428,5917.23,22,1.275,Decile 1
1,2,1726,1.294,1.919,3313.04,24,1.390,Decile 2
2,3,1725,1.566,2.065,3561.70,15,0.870,Decile 3
3,4,1726,1.833,2.469,4262.02,18,1.043,Decile 4
4,5,1726,2.131,0.940,1623.28,23,1.333,Decile 5
5,6,1725,2.469,3.132,5403.23,28,1.623,Decile 6
6,7,1726,2.860,1.613,2783.74,31,1.796,Decile 7
7,8,1725,3.408,3.415,5891.48,45,2.609,Decile 8
8,9,1726,4.316,3.094,5339.75,50,2.897,Decile 9
9,10,1726,7.392,4.795,8275.47,75,4.345,Decile 10



TOP VERSUS BOTTOM DECILE
Top-decile average observed value: R$4.79
Bottom-decile average observed value: R$3.43
Top-decile share of total subsequent value: 17.85%


In [20]:

# Investigate conditional spending among training customers who repeated
# This is an exploratory analysis, not model training

positive_train_mask = y_train > 0

positive_training_features = X_train.loc[
    positive_train_mask
].copy()

positive_training_value = y_train.loc[
    positive_train_mask
].copy()

print("POSITIVE-VALUE TRAINING POPULATION")
print(f"Repeat purchasers: {len(positive_training_value):,}")

print("\nSUBSEQUENT SPENDING DISTRIBUTION")

display(
    positive_training_value.describe(
        percentiles=[0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    ).round(2)
)

# Examine descriptive correlations for selected first-order features
candidate_spending_features = [
    "first_order_item_value",
    "first_order_item_count",
    "first_order_freight_value",
    "first_order_total_basket_value_log",
    "first_order_max_installments",
    "first_order_category_count"
]

conditional_spending_audit = (
    positive_training_features[candidate_spending_features]
    .assign(subsequent_value=positive_training_value)
)

spending_correlations = (
    conditional_spending_audit
    .corr(method="spearman")["subsequent_value"]
    .drop("subsequent_value")
    .sort_values(ascending=False)
)

print("\nSPEARMAN CORRELATION WITH SUBSEQUENT SPENDING")

display(
    spending_correlations
    .rename("Spearman Correlation")
    .round(3)
)


POSITIVE-VALUE TRAINING POPULATION
Repeat purchasers: 321

SUBSEQUENT SPENDING DISTRIBUTION


count     321.00
mean      122.99
std       214.73
min         7.80
10%        20.90
25%        38.90
50%        69.90
75%       126.99
90%       239.80
95%       369.70
99%       736.46
max      2589.00
Name: subsequent_60d_value, dtype: float64


SPEARMAN CORRELATION WITH SUBSEQUENT SPENDING


first_order_item_value                0.531
first_order_total_basket_value_log    0.523
first_order_freight_value             0.281
first_order_max_installments          0.260
first_order_category_count            0.073
first_order_item_count                0.068
Name: Spearman Correlation, dtype: float64

In [21]:

from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Train a simple conditional spending model using only customers
# with positive subsequent merchandise value

spending_feature = ["first_order_item_value"]

X_spending_train = X_train.loc[
    y_train > 0,
    spending_feature
].copy()

y_spending_train = y_train.loc[
    y_train > 0
].copy()

# Use log transformation to reduce the influence of large spending values
log_spending_target = np.log1p(y_spending_train)

# Fit preprocessing and regularised regression using training data only
conditional_spending_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("regressor", Ridge(alpha=10.0))
])

conditional_spending_model.fit(
    X_spending_train,
    log_spending_target
)

# Evaluate conditional spending among actual validation repeat purchasers
# This is diagnostic evaluation, not the final all-customer value assessment
positive_val_mask = y_val > 0

X_spending_val = X_val.loc[
    positive_val_mask,
    spending_feature
].copy()

y_spending_val = y_val.loc[
    positive_val_mask
].copy()

predicted_log_spending = conditional_spending_model.predict(
    X_spending_val
)

# Convert predictions back to merchandise value
predicted_spending = np.maximum(
    np.expm1(predicted_log_spending),
    0
)

# Compare against training-only constant spending baseline
constant_spending_predictions = np.full(
    len(y_spending_val),
    training_positive_mean
)

conditional_spending_results = pd.DataFrame([
    {
        "Model": "Training positive-value mean",
        "Mean Absolute Error (R$)": mean_absolute_error(
            y_spending_val,
            constant_spending_predictions
        ),
        "Root Mean Squared Error (R$)": np.sqrt(
            mean_squared_error(
                y_spending_val,
                constant_spending_predictions
            )
        ),
        "Average Predicted Value (R$)": constant_spending_predictions.mean(),
        "Average Actual Value (R$)": y_spending_val.mean()
    },
    {
        "Model": "Regularised conditional spending model",
        "Mean Absolute Error (R$)": mean_absolute_error(
            y_spending_val,
            predicted_spending
        ),
        "Root Mean Squared Error (R$)": np.sqrt(
            mean_squared_error(
                y_spending_val,
                predicted_spending
            )
        ),
        "Average Predicted Value (R$)": predicted_spending.mean(),
        "Average Actual Value (R$)": y_spending_val.mean()
    }
])

print("CONDITIONAL SPENDING MODEL EVALUATION")
print(f"Training repeat purchasers: {len(y_spending_train):,}")
print(f"Validation repeat purchasers: {len(y_spending_val):,}")

display(conditional_spending_results.round(2))


CONDITIONAL SPENDING MODEL EVALUATION
Training repeat purchasers: 321
Validation repeat purchasers: 331


,Model,Mean Absolute Error (R$),Root Mean Squared Error (R$),Average Predicted Value (R$),Average Actual Value (R$)
0,Training positive-value mean,103.15,195.63,122.99,140.09
1,Regularised conditional spending model,92.61,204.07,73.51,140.09


In [22]:

# Investigate retransformation bias in the conditional spending model
# Estimate the smearing adjustment using training data only

training_log_predictions = conditional_spending_model.predict(
    X_spending_train
)

training_log_residuals = (
    log_spending_target.to_numpy()
    - training_log_predictions
)

# Estimate the multiplicative adjustment on the (value + 1) scale
smearing_factor = np.mean(
    np.exp(training_log_residuals)
)

# Apply the training-derived correction to validation predictions
smearing_adjusted_spending = np.maximum(
    np.exp(predicted_log_spending) * smearing_factor - 1,
    0
)

# Compare the original model with the adjusted predictions
smearing_results = pd.DataFrame([
    {
        "Approach": "Unadjusted log-regression",
        "Mean Absolute Error (R$)": mean_absolute_error(
            y_spending_val, predicted_spending
        ),
        "Root Mean Squared Error (R$)": np.sqrt(
            mean_squared_error(
                y_spending_val, predicted_spending
            )
        ),
        "Average Predicted Spending (R$)": predicted_spending.mean(),
        "Observed Average Spending (R$)": y_spending_val.mean()
    },
    {
        "Approach": "Training-derived smearing adjustment",
        "Mean Absolute Error (R$)": mean_absolute_error(
            y_spending_val, smearing_adjusted_spending
        ),
        "Root Mean Squared Error (R$)": np.sqrt(
            mean_squared_error(
                y_spending_val, smearing_adjusted_spending
            )
        ),
        "Average Predicted Spending (R$)": (
            smearing_adjusted_spending.mean()
        ),
        "Observed Average Spending (R$)": y_spending_val.mean()
    }
])

print("TRAINING-DERIVED SMEARING FACTOR")
print(f"{smearing_factor:.4f}")

print("\nCONDITIONAL SPENDING RETRANSFORMATION COMPARISON")
display(smearing_results.round(2))


TRAINING-DERIVED SMEARING FACTOR
1.4638

CONDITIONAL SPENDING RETRANSFORMATION COMPARISON


,Approach,Mean Absolute Error (R$),Root Mean Squared Error (R$),Average Predicted Spending (R$),Observed Average Spending (R$)
0,Unadjusted log-regression,92.61,204.07,73.51,140.09
1,Training-derived smearing adjustment,94.20,195.09,108.07,140.09


In [23]:

# Evaluate the complete two-stage customer-value model
# All conditional spending predictions are generated for every
# validation customer, without using their actual repeat outcome

# Predict conditional spending for all validation customers
all_validation_log_spending = (
    conditional_spending_model.predict(
        X_val[spending_feature]
    )
)

# Apply the training-derived smearing adjustment
all_validation_conditional_spending = np.maximum(
    np.exp(all_validation_log_spending) * smearing_factor - 1,
    0
)

# Combine repeat probability with conditional spending
validation_two_stage_value = (
    validation_repeat_probability
    * all_validation_conditional_spending
)

# Compare against the earlier approaches
value_predictions = {
    "Constant training mean": np.full(
        len(y_val),
        training_mean_value
    ),
    "Probability × constant spending": validation_expected_value,
    "Probability × adjusted spending": validation_two_stage_value
}

two_stage_results = []

for model_name, predictions in value_predictions.items():
    two_stage_results.append({
        "Model": model_name,
        "Mean Absolute Error (R$)": mean_absolute_error(
            y_val, predictions
        ),
        "Root Mean Squared Error (R$)": np.sqrt(
            mean_squared_error(y_val, predictions)
        ),
        "Average Predicted Value (R$)": predictions.mean(),
        "Observed Average Value (R$)": y_val.mean(),
        "Average Prediction Bias (R$)": (
            predictions.mean() - y_val.mean()
        )
    })

two_stage_comparison = pd.DataFrame(two_stage_results)

# Validate prediction integrity
assert len(validation_two_stage_value) == len(y_val)
assert np.isfinite(validation_two_stage_value).all()
assert (validation_two_stage_value >= 0).all()

print("COMPLETE TWO-STAGE MODEL COMPARISON")

display(two_stage_comparison.round(3))


COMPLETE TWO-STAGE MODEL COMPARISON


,Model,Mean Absolute Error (R$),Root Mean Squared Error (R$),Average Predicted Value (R$),Observed Average Value (R$),Average Prediction Bias (R$)
0,Constant training mean,5.296,33.131,2.713,2.687,0.025
1,Probability × constant spending,5.352,33.134,2.820,2.687,0.133
2,Probability × adjusted spending,6.255,152.844,3.714,2.687,1.027


In [24]:

# Diagnose unusually large predictions in the complete two-stage model
# Identify whether extreme values are driving poor monetary accuracy

prediction_diagnostics = pd.DataFrame({
    "first_order_item_value": X_val[
        "first_order_item_value"
    ].to_numpy(),
    "repeat_probability": validation_repeat_probability,
    "predicted_conditional_spending": all_validation_conditional_spending,
    "predicted_customer_value": validation_two_stage_value,
    "actual_customer_value": y_val.to_numpy()
})

# Calculate prediction errors
prediction_diagnostics["prediction_error"] = (
    prediction_diagnostics["predicted_customer_value"]
    - prediction_diagnostics["actual_customer_value"]
)

prediction_diagnostics["squared_error"] = (
    prediction_diagnostics["prediction_error"] ** 2
)

print("PREDICTION DISTRIBUTION")

display(
    prediction_diagnostics[
        [
            "first_order_item_value",
            "predicted_conditional_spending",
            "predicted_customer_value",
            "actual_customer_value"
        ]
    ].describe(
        percentiles=[0.50, 0.90, 0.95, 0.99, 0.999]
    ).round(2)
)

print("\nTEN CUSTOMERS WITH LARGEST SQUARED ERRORS")

display(
    prediction_diagnostics.sort_values(
        "squared_error",
        ascending=False
    ).head(10).round(2)
)

print("\nSQUARED ERROR CONCENTRATION")

total_squared_error = (
    prediction_diagnostics["squared_error"].sum()
)

for n in [1, 5, 10, 50]:
    error_share = (
        prediction_diagnostics["squared_error"]
        .nlargest(n)
        .sum()
        / total_squared_error * 100
    )

    print(
        f"Top {n} customers: {error_share:.2f}% "
        "of total squared error"
    )


PREDICTION DISTRIBUTION


,first_order_item_value,predicted_conditional_spending,predicted_customer_value,actual_customer_value
count,17060.00,17257.00,17257.00,17257.00
mean,137.48,120.00,3.71,2.69
std,196.72,744.08,149.21,33.13
min,3.85,93.44,0.34,0.00
50%,86.99,102.85,1.99,0.00
90%,271.08,127.93,4.28,0.00
95%,399.50,148.94,5.49,0.00
99%,978.74,291.01,9.66,78.43
99.9%,2075.58,1052.84,38.26,349.62
max,5934.60,97259.88,19596.38,1599.00



TEN CUSTOMERS WITH LARGEST SQUARED ERRORS


,first_order_item_value,repeat_probability,predicted_conditional_spending,predicted_customer_value,actual_customer_value,prediction_error,squared_error
4309,5934.60,0.20,97259.88,19596.38,0.00,19596.38,3.840181e+08
10838,64.00,0.02,100.33,1.60,1599.00,-1597.40,2.551680e+06
8166,394.65,0.01,148.17,1.12,1375.65,-1374.53,1.889341e+06
5219,21.60,0.03,95.43,2.59,1150.00,-1147.41,1.316561e+06
14152,NaN,0.02,101.03,2.03,1128.00,-1125.97,1.267800e+06
1208,235.92,0.01,122.90,1.02,998.00,-996.98,9.939708e+05
14439,190.83,0.01,116.53,1.47,950.00,-948.53,8.997084e+05
9372,95.00,0.01,104.07,1.39,949.90,-948.51,8.996786e+05
14974,20.00,0.02,95.24,1.99,816.90,-814.91,6.640717e+05
14905,NaN,0.02,101.03,1.97,725.00,-723.03,5.227777e+05



SQUARED ERROR CONCENTRATION
Top 1 customers: 95.26% of total squared error
Top 5 customers: 97.00% of total squared error
Top 10 customers: 97.99% of total squared error
Top 50 customers: 99.26% of total squared error


In [25]:

# Investigate whether extreme validation purchases fall outside
# the first-order value range used to train the spending model

training_initial_values = (
    X_spending_train["first_order_item_value"].dropna()
)

validation_initial_values = (
    X_val["first_order_item_value"].dropna()
)

feature_range_audit = pd.DataFrame({
    "Population": [
        "Training repeat purchasers",
        "All development-validation customers",
        "Validation repeat purchasers"
    ],
    "Customers with recorded initial value": [
        len(training_initial_values),
        len(validation_initial_values),
        X_spending_val["first_order_item_value"].notna().sum()
    ],
    "Median Initial Value (R$)": [
        training_initial_values.median(),
        validation_initial_values.median(),
        X_spending_val["first_order_item_value"].median()
    ],
    "99th Percentile Initial Value (R$)": [
        training_initial_values.quantile(0.99),
        validation_initial_values.quantile(0.99),
        X_spending_val["first_order_item_value"].quantile(0.99)
    ],
    "Maximum Initial Value (R$)": [
        training_initial_values.max(),
        validation_initial_values.max(),
        X_spending_val["first_order_item_value"].max()
    ]
})

display(feature_range_audit.round(2))

print("\nVALIDATION CUSTOMERS BEYOND TRAINING POSITIVE RANGE")

training_max = training_initial_values.max()

beyond_training_range = (
    X_val["first_order_item_value"] > training_max
)

print(
    "Training repeat-purchaser maximum:",
    f"R${training_max:,.2f}"
)

print(
    "Validation customers exceeding this maximum:",
    f"{beyond_training_range.sum():,}"
)

print(
    "Maximum first-order value in validation:",
    f"R${validation_initial_values.max():,.2f}"
)


,Population,Customers with recorded initial value,Median Initial Value (R$),99th Percentile Initial Value (R$),Maximum Initial Value (R$)
0,Training repeat purchasers,315,69.90,726.82,4799.0
1,All development-validation customers,17060,86.99,978.74,5934.6
2,Validation repeat purchasers,326,70.90,689.00,1141.0



VALIDATION CUSTOMERS BEYOND TRAINING POSITIVE RANGE
Training repeat-purchaser maximum: R$4,799.00
Validation customers exceeding this maximum: 1
Maximum first-order value in validation: R$5,934.60


In [26]:

from sklearn.preprocessing import FunctionTransformer

# Test log-transformed initial merchandise value as the predictor
# Fit all preprocessing steps using positive-value training customers only

log_input_spending_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("log_transform", FunctionTransformer(
        np.log1p,
        validate=False
    )),
    ("scaler", StandardScaler()),
    ("regressor", Ridge(alpha=10.0))
])

log_input_spending_model.fit(
    X_spending_train,
    log_spending_target
)

# Generate predictions for positive-value validation customers
log_input_val_predictions = log_input_spending_model.predict(
    X_spending_val
)

# Estimate a new training-derived smearing adjustment
log_input_train_predictions = log_input_spending_model.predict(
    X_spending_train
)

log_input_residuals = (
    log_spending_target.to_numpy()
    - log_input_train_predictions
)

log_input_smearing_factor = np.exp(
    log_input_residuals
).mean()

# Convert log predictions back to monetary values
log_input_adjusted_spending = np.maximum(
    np.exp(log_input_val_predictions)
    * log_input_smearing_factor
    - 1,
    0
)

# Compare conditional spending predictions
log_input_comparison = pd.DataFrame([
    {
        "Model": "Original adjusted spending model",
        "Mean Absolute Error (R$)": mean_absolute_error(
            y_spending_val, smearing_adjusted_spending
        ),
        "Root Mean Squared Error (R$)": np.sqrt(
            mean_squared_error(
                y_spending_val, smearing_adjusted_spending
            )
        ),
        "Average Predicted Spending (R$)": smearing_adjusted_spending.mean()
    },
    {
        "Model": "Log-input adjusted spending model",
        "Mean Absolute Error (R$)": mean_absolute_error(
            y_spending_val, log_input_adjusted_spending
        ),
        "Root Mean Squared Error (R$)": np.sqrt(
            mean_squared_error(
                y_spending_val, log_input_adjusted_spending
            )
        ),
        "Average Predicted Spending (R$)": log_input_adjusted_spending.mean()
    }
])

print("LOG-INPUT SPENDING MODEL EVALUATION")
print(f"Training-derived smearing factor: {log_input_smearing_factor:.4f}")
print(f"Observed mean spending: R${y_spending_val.mean():.2f}")

display(log_input_comparison.round(2))


LOG-INPUT SPENDING MODEL EVALUATION
Training-derived smearing factor: 1.4136
Observed mean spending: R$140.09


,Model,Mean Absolute Error (R$),Root Mean Squared Error (R$),Average Predicted Spending (R$)
0,Original adjusted spending model,94.20,195.09,108.07
1,Log-input adjusted spending model,93.84,191.91,121.98


In [27]:

# Evaluate the revised two-stage customer-value model
# Predict conditional spending for every development-validation customer

# Generate log-spending predictions using the improved model
all_validation_log_input_predictions = (
    log_input_spending_model.predict(
        X_val[spending_feature]
    )
)

# Apply the training-derived retransformation adjustment
all_validation_log_input_spending = np.maximum(
    np.exp(all_validation_log_input_predictions)
    * log_input_smearing_factor
    - 1,
    0
)

# Combine conditional spending with repeat-purchase probability
validation_revised_two_stage_value = (
    validation_repeat_probability
    * all_validation_log_input_spending
)

# Compare all approaches on the same development-validation population
revised_predictions = {
    "Predict zero": np.zeros(len(y_val)),
    "Constant training mean": np.full(
        len(y_val),
        training_mean_value
    ),
    "Probability x constant spending": validation_expected_value,
    "Original adjusted two-stage": validation_two_stage_value,
    "Revised log-input two-stage": validation_revised_two_stage_value
}

revised_comparison_results = []

for model_name, predictions in revised_predictions.items():
    revised_comparison_results.append({
        "Model": model_name,
        "Mean Absolute Error (R$)": mean_absolute_error(
            y_val, predictions
        ),
        "Root Mean Squared Error (R$)": np.sqrt(
            mean_squared_error(y_val, predictions)
        ),
        "Average Predicted Value (R$)": predictions.mean(),
        "Observed Average Value (R$)": y_val.mean(),
        "Average Prediction Bias (R$)": (
            predictions.mean() - y_val.mean()
        )
    })

revised_comparison = pd.DataFrame(revised_comparison_results)

# Check prediction integrity
assert len(validation_revised_two_stage_value) == len(y_val)
assert np.isfinite(validation_revised_two_stage_value).all()
assert (validation_revised_two_stage_value >= 0).all()

print("REVISED TWO-STAGE MODEL COMPARISON")

display(revised_comparison.round(3))

print("\nREVISED MODEL PREDICTION DISTRIBUTION")

display(
    pd.Series(
        validation_revised_two_stage_value,
        name="predicted_subsequent_value"
    ).describe(
        percentiles=[0.50, 0.90, 0.95, 0.99, 0.999]
    ).round(2)
)


REVISED TWO-STAGE MODEL COMPARISON


,Model,Mean Absolute Error (R$),Root Mean Squared Error (R$),Average Predicted Value (R$),Observed Average Value (R$),Average Prediction Bias (R$)
0,Predict zero,2.687,33.240,0.000,2.687,-2.687
1,Constant training mean,5.296,33.131,2.713,2.687,0.025
2,Probability x constant spending,5.352,33.134,2.820,2.687,0.133
3,Original adjusted two-stage,6.255,152.844,3.714,2.687,1.027
4,Revised log-input two-stage,5.319,33.266,2.799,2.687,0.112



REVISED MODEL PREDICTION DISTRIBUTION


count    17257.00
mean         2.80
std          4.34
min          0.48
50%          2.02
90%          4.75
95%          6.23
99%         11.73
99.9%       42.08
max        249.20
Name: predicted_subsequent_value, dtype: float64

In [28]:

# Compare customer-value ranking across the two modelling approaches
# Use development-validation outcomes only

ranking_models = {
    "Probability x constant spending": validation_expected_value,
    "Revised log-input two-stage": validation_revised_two_stage_value
}

ranking_comparison_results = []

for model_name, predictions in ranking_models.items():

    ranking_data = pd.DataFrame({
        "predicted_value": predictions,
        "actual_value": y_val.to_numpy()
    })

    # Rank customers into ten approximately equal groups
    ranking_data["decile"] = pd.qcut(
        ranking_data["predicted_value"].rank(method="first"),
        q=10,
        labels=range(1, 11)
    ).astype(int)

    top_decile_data = ranking_data.loc[
        ranking_data["decile"] == 10
    ]

    bottom_decile_data = ranking_data.loc[
        ranking_data["decile"] == 1
    ]

    ranking_comparison_results.append({
        "Model": model_name,
        "Top Decile Average Actual Value (R$)": (
            top_decile_data["actual_value"].mean()
        ),
        "Bottom Decile Average Actual Value (R$)": (
            bottom_decile_data["actual_value"].mean()
        ),
        "Top Decile Share of Total Actual Value (%)": (
            100
            * top_decile_data["actual_value"].sum()
            / ranking_data["actual_value"].sum()
        ),
        "Top Decile Positive-Value Rate (%)": (
            100
            * (top_decile_data["actual_value"] > 0).mean()
        )
    })

ranking_model_comparison = pd.DataFrame(
    ranking_comparison_results
)

print("CUSTOMER-VALUE RANKING COMPARISON")

display(ranking_model_comparison.round(3))


CUSTOMER-VALUE RANKING COMPARISON


,Model,Top Decile Average Actual Value (R$),Bottom Decile Average Actual Value (R$),Top Decile Share of Total Actual Value (%),Top Decile Positive-Value Rate (%)
0,Probability x constant spending,4.795,3.428,17.846,4.345
1,Revised log-input two-stage,4.680,1.486,17.419,3.534


In [29]:

# Evaluate how much actual subsequent merchandise value is captured
# when selecting the highest-ranked customers in development validation

selection_predictions = {
    "Probability x constant spending": validation_expected_value,
    "Revised log-input two-stage": validation_revised_two_stage_value
}

selection_results = []

actual_values = y_val.to_numpy()
total_actual_value = actual_values.sum()

for model_name, predictions in selection_predictions.items():

    selection_data = pd.DataFrame({
        "predicted_value": predictions,
        "actual_value": actual_values
    }).sort_values(
        "predicted_value",
        ascending=False,
        kind="stable"
    )

    for selection_share in [0.10, 0.20, 0.30]:

        selected_count = int(
            np.ceil(len(selection_data) * selection_share)
        )

        selected = selection_data.head(selected_count)

        selection_results.append({
            "Model": model_name,
            "Selected Customers (%)": int(selection_share * 100),
            "Customers Selected": selected_count,
            "Share of Actual Value Captured (%)": (
                100 * selected["actual_value"].sum()
                / total_actual_value
            ),
            "Average Actual Value (R$)": (
                selected["actual_value"].mean()
            ),
            "Positive-Value Rate (%)": (
                100 * (selected["actual_value"] > 0).mean()
            )
        })

selection_comparison = pd.DataFrame(selection_results)

print("CUSTOMER-VALUE CAPTURE BY SELECTION SIZE")

display(
    selection_comparison.sort_values(
        ["Selected Customers (%)", "Model"]
    ).round(3)
)


CUSTOMER-VALUE CAPTURE BY SELECTION SIZE


,Model,Selected Customers (%),Customers Selected,Share of Actual Value Captured (%),Average Actual Value (R$),Positive-Value Rate (%)
0,Probability x constant spending,10,1726,17.846,4.795,4.345
3,Revised log-input two-stage,10,1726,17.419,4.680,3.534
1,Probability x constant spending,20,3452,29.362,3.944,3.621
4,Revised log-input two-stage,20,3452,28.112,3.776,2.868
2,Probability x constant spending,30,5178,42.067,3.767,3.283
5,Revised log-input two-stage,30,5178,41.228,3.692,2.839


In [30]:

# Retrospective temporal evaluation on later historical customers
# Reuse trained models without refitting or changing their parameters

# Probability predictions for later customers
evaluation_repeat_probability = (
    repeat_probability_model.predict_proba(X_eval)[:, 1]
)

# Simple customer-specific expected value
evaluation_probability_value = (
    evaluation_repeat_probability * training_positive_mean
)

# Revised conditional spending predictions
evaluation_log_spending = (
    log_input_spending_model.predict(
        X_eval[spending_feature]
    )
)

evaluation_conditional_spending = np.maximum(
    np.exp(evaluation_log_spending)
    * log_input_smearing_factor
    - 1,
    0
)

evaluation_revised_value = (
    evaluation_repeat_probability
    * evaluation_conditional_spending
)

# Compare models on later historical customers
evaluation_predictions = {
    "Predict zero": np.zeros(len(y_eval)),
    "Constant training mean": np.full(
        len(y_eval),
        training_mean_value
    ),
    "Probability x constant spending": evaluation_probability_value,
    "Revised log-input two-stage": evaluation_revised_value
}

retrospective_results = []

for model_name, predictions in evaluation_predictions.items():
    retrospective_results.append({
        "Model": model_name,
        "Mean Absolute Error (R$)": mean_absolute_error(
            y_eval, predictions
        ),
        "Root Mean Squared Error (R$)": np.sqrt(
            mean_squared_error(y_eval, predictions)
        ),
        "Average Predicted Value (R$)": predictions.mean(),
        "Observed Average Value (R$)": y_eval.mean(),
        "Average Prediction Bias (R$)": (
            predictions.mean() - y_eval.mean()
        )
    })

retrospective_comparison = pd.DataFrame(retrospective_results)

# Validate prediction integrity
assert len(evaluation_probability_value) == len(y_eval)
assert len(evaluation_revised_value) == len(y_eval)
assert np.isfinite(evaluation_probability_value).all()
assert np.isfinite(evaluation_revised_value).all()

print("RETROSPECTIVE TEMPORAL MODEL COMPARISON")
print(f"Evaluation customers: {len(y_eval):,}")
print(f"Customers with positive value: {(y_eval > 0).sum():,}")

display(retrospective_comparison.round(3))


RETROSPECTIVE TEMPORAL MODEL COMPARISON
Evaluation customers: 13,649
Customers with positive value: 199


,Model,Mean Absolute Error (R$),Root Mean Squared Error (R$),Average Predicted Value (R$),Observed Average Value (R$),Average Prediction Bias (R$)
0,Predict zero,2.323,31.401,0.000,2.323,-2.323
1,Constant training mean,4.956,31.317,2.713,2.323,0.390
2,Probability x constant spending,4.722,31.380,2.481,2.323,0.158
3,Revised log-input two-stage,4.739,31.431,2.498,2.323,0.175


In [31]:

# Evaluate customer-value ranking on later historical customers
# Compare the same two models without retraining or tuning

retrospective_ranking_models = {
    "Probability x constant spending": evaluation_probability_value,
    "Revised log-input two-stage": evaluation_revised_value
}

actual_eval_values = y_eval.to_numpy()
total_eval_value = actual_eval_values.sum()

retrospective_ranking_results = []

for model_name, predictions in retrospective_ranking_models.items():

    ranking_data = pd.DataFrame({
        "predicted_value": predictions,
        "actual_value": actual_eval_values
    })

    # Rank customers by predicted monetary value
    ranking_data = ranking_data.sort_values(
        "predicted_value",
        ascending=False,
        kind="stable"
    )

    for selection_share in [0.10, 0.20, 0.30]:

        selected_count = int(
            np.ceil(len(ranking_data) * selection_share)
        )

        selected = ranking_data.head(selected_count)

        retrospective_ranking_results.append({
            "Model": model_name,
            "Selected Customers (%)": int(selection_share * 100),
            "Customers Selected": selected_count,
            "Share of Actual Value Captured (%)": (
                100 * selected["actual_value"].sum()
                / total_eval_value
            ),
            "Average Actual Value (R$)": (
                selected["actual_value"].mean()
            ),
            "Positive-Value Rate (%)": (
                100 * (selected["actual_value"] > 0).mean()
            )
        })

retrospective_ranking_summary = pd.DataFrame(
    retrospective_ranking_results
)

# Validate evaluation coverage
assert len(y_eval) == 13_649
assert (y_eval > 0).sum() == 199
assert total_eval_value > 0

print("RETROSPECTIVE CUSTOMER-VALUE RANKING")

display(
    retrospective_ranking_summary.sort_values(
        ["Selected Customers (%)", "Model"]
    ).round(3)
)


RETROSPECTIVE CUSTOMER-VALUE RANKING


,Model,Selected Customers (%),Customers Selected,Share of Actual Value Captured (%),Average Actual Value (R$),Positive-Value Rate (%)
0,Probability x constant spending,10,1365,10.332,2.400,2.491
3,Revised log-input two-stage,10,1365,24.574,5.707,1.978
1,Probability x constant spending,20,2730,15.949,1.852,2.088
4,Revised log-input two-stage,20,2730,34.629,4.021,1.538
2,Probability x constant spending,30,4095,22.631,1.752,1.758
5,Revised log-input two-stage,30,4095,45.828,3.548,1.807


In [32]:

# Investigate the source of the revised model's stronger
# retrospective monetary-value capture
# Use existing predictions without retraining or changing the models

top_decile_diagnostics = []

for model_name, predictions in retrospective_ranking_models.items():

    ranked_customers = pd.DataFrame({
        "predicted_value": predictions,
        "actual_value": y_eval.to_numpy()
    }).sort_values(
        "predicted_value",
        ascending=False,
        kind="stable"
    )

    selected_count = int(np.ceil(len(ranked_customers) * 0.10))
    selected = ranked_customers.head(selected_count)

    positive_selected = selected.loc[
        selected["actual_value"] > 0
    ]

    total_selected_value = selected["actual_value"].sum()

    top_decile_diagnostics.append({
        "Model": model_name,
        "Selected Customers": len(selected),
        "Customers with Positive Value": len(positive_selected),
        "Average Value Among Repeat Purchasers (R$)": (
            positive_selected["actual_value"].mean()
        ),
        "Median Value Among Repeat Purchasers (R$)": (
            positive_selected["actual_value"].median()
        ),
        "Largest Actual Customer Value (R$)": (
            selected["actual_value"].max()
        ),
        "Top 5 Actual Purchasers' Share of Selected Value (%)": (
            100
            * selected["actual_value"].nlargest(5).sum()
            / total_selected_value
            if total_selected_value > 0
            else np.nan
        )
    })

top_decile_diagnostic_summary = pd.DataFrame(
    top_decile_diagnostics
)

print("TOP-DECILE MONETARY-VALUE DIAGNOSTICS")

display(top_decile_diagnostic_summary.round(2))


TOP-DECILE MONETARY-VALUE DIAGNOSTICS


,Model,Selected Customers,Customers with Positive Value,Average Value Among Repeat Purchasers (R$),Median Value Among Repeat Purchasers (R$),Largest Actual Customer Value (R$),Top 5 Actual Purchasers' Share of Selected Value (%)
0,Probability x constant spending,1365,34,96.33,49.99,419.8,55.49
1,Revised log-input two-stage,1365,27,288.53,215.60,1054.0,49.58


In [33]:

# Assess customer-value ranking stability across retrospective months
# Keep model predictions unchanged and evaluate each month separately

retrospective_monthly_data = pd.DataFrame({
    "purchase_month": modelling_data.loc[
        evaluation_mask,
        "order_purchase_timestamp"
    ].dt.to_period("M").astype(str).to_numpy(),
    "actual_value": y_eval.to_numpy(),
    "probability_based_value": evaluation_probability_value,
    "revised_two_stage_value": evaluation_revised_value
})

monthly_ranking_results = []

for month, month_data in retrospective_monthly_data.groupby(
    "purchase_month"
):
    for model_name, prediction_column in [
        ("Probability x constant spending", "probability_based_value"),
        ("Revised log-input two-stage", "revised_two_stage_value")
    ]:
        ranked = month_data.sort_values(
            prediction_column,
            ascending=False,
            kind="stable"
        )

        selected_count = int(np.ceil(len(ranked) * 0.10))
        selected = ranked.head(selected_count)

        total_month_value = month_data["actual_value"].sum()

        monthly_ranking_results.append({
            "Purchase Month": month,
            "Model": model_name,
            "Customers": len(month_data),
            "Positive-Value Customers": (
                month_data["actual_value"] > 0
            ).sum(),
            "Top 10% Customers": selected_count,
            "Top 10% Value Capture (%)": (
                100 * selected["actual_value"].sum()
                / total_month_value
                if total_month_value > 0
                else np.nan
            ),
            "Top 10% Positive-Value Rate (%)": (
                100 * (selected["actual_value"] > 0).mean()
            )
        })

monthly_ranking_comparison = pd.DataFrame(
    monthly_ranking_results
)

print("MONTHLY RETROSPECTIVE RANKING STABILITY")

display(
    monthly_ranking_comparison.sort_values(
        ["Purchase Month", "Model"]
    ).round(2)
)


MONTHLY RETROSPECTIVE RANKING STABILITY


,Purchase Month,Model,Customers,Positive-Value Customers,Top 10% Customers,Top 10% Value Capture (%),Top 10% Positive-Value Rate (%)
0,2018-04,Probability x constant spending,6711,104,672,8.20,2.53
1,2018-04,Revised log-input two-stage,6711,104,672,24.73,2.23
2,2018-05,Probability x constant spending,6621,90,663,12.18,2.26
3,2018-05,Revised log-input two-stage,6621,90,663,26.07,1.96
4,2018-06,Probability x constant spending,317,5,32,8.48,6.25
5,2018-06,Revised log-input two-stage,317,5,32,0.00,0.00



## Predictive Customer Value: Model Evaluation and Selection

### Business Question

Can we estimate additional customer merchandise value within 60 days of the first purchase, and does modelling purchase amount improve customer prioritisation beyond repeat-purchase probability?

### Approaches Evaluated

1. Zero-value and constant-mean baselines.
2. Customer-specific repeat-purchase probability multiplied by average spending among repeat purchasers.
3. Two-stage modelling combining repeat-purchase probability with customer-specific conditional spending.
4. A revised two-stage approach using log-transformed initial purchase value and training-derived retransformation adjustment.

### Key Findings

**1. Accurate individual monetary prediction remains challenging.**

Only 1,589 of 78,100 customers generated additional delivered merchandise value within 60 days. This highly sparse target makes simple zero-value and constant-mean predictions difficult to outperform on conventional error metrics.

**2. The original conditional spending model was unstable.**

One validation customer received an expected-value prediction of R$19,596.38 despite generating no subsequent value, accounting for 95.26% of the original adjusted model's squared error.

**3. Log-transforming initial merchandise value improved prediction stability.**

The revised model reduced the largest predicted expected value to R$249.20 and substantially reduced monetary prediction error relative to the original adjusted model.

**4. Customer-specific models did not consistently outperform simple monetary baselines.**

On development validation, the constant training-mean baseline achieved a root mean squared error of R$33.131, compared with R$33.266 for the revised two-stage model.

On retrospective evaluation, the constant baseline achieved R$31.317, compared with R$31.431 for the revised two-stage model.

**5. The revised two-stage model showed stronger retrospective monetary-value capture.**

In April 2018, its highest-ranked 10% captured 24.73% of subsequent merchandise value, compared with 8.20% for the probability-based benchmark.

In May 2018, the corresponding figures were 26.07% and 12.18%.

However, development-validation ranking favoured the simpler probability-based model.

### Provisional Model Decision

Retain both models for further investigation.

The revised two-stage model is the leading experimental candidate for monetary-value ranking based on April–May retrospective performance.

The probability-based model remains an important benchmark for repeat-purchase identification and simpler customer prioritisation.

Neither approach is approved for production monetary forecasting or retention intervention decisions.

### Limitations

- The positive-spending training population contains only 321 customers.
- Monetary outcomes are highly skewed and concentrated among relatively few customers.
- The conditional spending model assumes a stable relationship between first-order and subsequent merchandise value.
- Feature availability at the intended post-checkout prediction point remains partly assumed.
- The retrospective evaluation population was previously examined and is not an untouched independent holdout.
- Comparing multiple approaches using historical evaluation outcomes introduces model-selection bias.
- The results do not establish profitability or incremental impact from customer targeting.

### Next Development Requirements

- Assess stability through additional chronological development checks.
- Investigate monetary calibration and uncertainty.
- Evaluate whether the ranking advantage persists under genuinely new data.
- Establish the commercial decision and intervention economics before deployment.

### Conclusion

Predictive customer-value modelling adds a potentially useful monetary perspective to the Customer Intelligence Platform, but the current evidence supports experimental customer ranking rather than reliable customer lifetime value forecasting.

The results demonstrate why predicting whether customers return and predicting how much they spend must be evaluated as distinct objectives.
